In [4]:
import numpy as np
from SelfAttention import LSTM, Layer_Dense, SelfAttention

np.random.seed(0)

In [5]:
# synthetic data helpers
def generate_sample(V=10, T=6):
    seq = np.random.randint(1, V-1, size=T)
    tgt = seq[::-1]
    return seq, tgt

In [6]:
def one_hot_sequence(seq, V):
    T = len(seq)
    out = np.zeros((T, V))
    out[np.arange(T), seq] = 1.0
    return out

In [7]:
# model hyper‑params
V = 10             # vocabulary size
T = 6              # sequence length
n_neurons = 32
lr = 1e-2
epochs = 5
batch_size = 16

In [8]:
# modules
encoder = LSTM(n_neurons, input_dim=V)
self_attn = SelfAttention(n_neurons)
output_layer = Layer_Dense(n_neurons, V)

In [9]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(x)
    return exp / np.sum(exp, axis=-1, keepdims=True)


In [10]:
# training loop
for ep in range(epochs):
    loss_sum = 0
    for b in range(batch_size):
        s, t = generate_sample(V=V, T=T)
        x = one_hot_sequence(s, V)
        y = np.array(t)

        # encode sequence with LSTM
        encoder.forward(x)
        H_enc = np.array(encoder.H[1:]).reshape(T, n_neurons)

        # apply self‑attention to encoder outputs
        H_sa, alpha = self_attn.forward(H_enc)

        # outputs & loss
        logits = output_layer.forward(H_sa)
        probs = softmax(logits)
        logp = -np.log(probs[np.arange(T), y] + 1e-12)
        loss = np.mean(logp)
        loss_sum += loss

        # backward
        dlogits = probs.copy()
        dlogits[np.arange(T), y] -= 1
        dlogits /= T
        output_layer.backward(dlogits)
        dH_sa = output_layer.dinputs
        dH_enc = self_attn.backward(dH_sa)
        encoder.backward(dH_enc)

        # SGD updates (simple)
        encoder.Uf -= lr * encoder.dUf
        encoder.Ui -= lr * encoder.dUi
        encoder.Uo -= lr * encoder.dUo
        encoder.Ug -= lr * encoder.dUg
        encoder.Wf -= lr * encoder.dWf
        encoder.Wi -= lr * encoder.dWi
        encoder.Wo -= lr * encoder.dWo
        encoder.Wg -= lr * encoder.dWg
        encoder.bf -= lr * encoder.dbf
        encoder.bi -= lr * encoder.dbi
        encoder.bo -= lr * encoder.dbo
        encoder.bg -= lr * encoder.dbg

        self_attn.W_q -= lr * self_attn.dW_q
        self_attn.W_k -= lr * self_attn.dW_k
        self_attn.W_v -= lr * self_attn.dW_v
        self_attn.W_o -= lr * self_attn.dW_o

        output_layer.weights -= lr * output_layer.dweights
        output_layer.biases  -= lr * output_layer.dbiases

    print(f"Epoch {ep+1}: mean loss = {loss_sum/batch_size:.4f}")


Epoch 1: mean loss = 2.2858
Epoch 2: mean loss = 2.2922
Epoch 3: mean loss = 2.2600
Epoch 4: mean loss = 2.2361
Epoch 5: mean loss = 2.2468


In [11]:
# simple inference sample
s, _ = generate_sample(V=V, T=T)
x = one_hot_sequence(s, V)
encoder.forward(x)
H_enc = np.array(encoder.H[1:]).reshape(T, n_neurons)
H_sa, alpha = self_attn.forward(H_enc)
logits = output_layer.forward(H_sa)
pred = np.argmax(softmax(logits), axis=-1)
print("Input:", s)
print("Predicted reversed indices:", pred)

Input: [4 8 2 5 5 4]
Predicted reversed indices: [3 3 3 3 3 3]
